In [3]:
from pathlib import Path
import pandas as pd

# Use raw string (r"") so backslashes in Windows path don't cause escape errors
folder_path = Path(r"C:\Users\srinu\OneDrive\Documents\Mine\Projects\archive")

# Check what files are inside this folder first
print("Files in folder:", [f.name for f in folder_path.iterdir()])

# Load CSV (change to pd.read_excel(...) if it's an Excel file)
file_path = folder_path / "DataCoSupplyChainDataset.csv"
df = pd.read_csv(file_path, encoding="latin1")

print(f"\nLoaded successfully! Shape: {df.shape}")
print("\nMissing values (top 10 columns with nulls):")
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False).head(10))

Files in folder: ['DataCoSupplyChainDataset.csv', 'DescriptionDataCoSupplyChain.csv', 'tokenized_access_logs.csv']

Loaded successfully! Shape: (180519, 53)

Missing values (top 10 columns with nulls):
Product Description    180519
Order Zipcode          155679
Customer Lname              8
Customer Zipcode            3
dtype: int64


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

folder_path = Path(r"C:\Users\srinu\OneDrive\Documents\Mine\Projects\archive")
df = pd.read_csv(folder_path / "DataCoSupplyChainDataset.csv", encoding="latin1")

# 1. Drop completely empty & unneeded columns
cols_to_drop = [
    "Product Description",
    "Product Image",
    "Customer Email",
    "Customer Password",
    "Customer Street",
    "Order Zipcode",
]
existing_drops = [c for c in cols_to_drop if c in df.columns]
df_clean = df.drop(columns=existing_drops).copy()

# 2. Fill minor missing values & drop exact duplicate rows
if "Customer Lname" in df_clean.columns:
  df_clean["Customer Lname"] = df_clean["Customer Lname"].fillna("Unknown")
if "Customer Zipcode" in df_clean.columns:
  df_clean["Customer Zipcode"] = (
      df_clean["Customer Zipcode"].fillna(0).astype(int).astype(str)
  )

before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before_rows - len(df_clean)} exact duplicate rows.")

# 3. Convert order date to datetime & extract Year-Month
df_clean["order date (DateOrders)"] = pd.to_datetime(
    df_clean["order date (DateOrders)"], errors="coerce"
)
df_clean["Order_YearMonth"] = (
    df_clean["order date (DateOrders)"].dt.to_period("M").astype(str)
)

# 4. Create new business metrics for analysis & Tableau
df_clean["Shipping_Delay_Days"] = (
    df_clean["Days for shipping (real)"]
    - df_clean["Days for shipment (scheduled)"]
)

df_clean["Profit_Margin_Pct"] = np.where(
    df_clean["Sales"] != 0,
    np.round((df_clean["Benefit per order"] / df_clean["Sales"]) * 100, 2),
    0.0,
)

df_clean["Delivery_Performance"] = np.where(
    df_clean["Shipping_Delay_Days"] > 0,
    "Delayed",
    np.where(df_clean["Shipping_Delay_Days"] < 0, "Early", "On Time"),
)

# 5. Save cleaned dataset as CSV
output_path = folder_path / "DataCo_Cleaned_SupplyChain.csv"
df_clean.to_csv(output_path, index=False)

print(f"Cleaned shape: {df_clean.shape}")
print("Saved clean file to:", output_path)
print("\nDelivery Performance counts:")
print(df_clean["Delivery_Performance"].value_counts())

Removed 0 exact duplicate rows.
Cleaned shape: (180519, 51)
Saved clean file to: C:\Users\srinu\OneDrive\Documents\Mine\Projects\archive\DataCo_Cleaned_SupplyChain.csv

Delivery Performance counts:
Delivery_Performance
Delayed    103400
Early       43366
On Time     33753
Name: count, dtype: int64
